### Self Supervised Pre Training

1. Load Benign Windows
2. Build a DataLoader
3. Mask 30%
4. Train a 1D CNN to reconstruct
5. Save the pretrained encoder


In [12]:
import torch 
from torch.utils.data import DataLoader, Dataset
import pandas as pd 
import numpy as np 
import torch.nn as nn 
import torch.optim as optim 
from pathlib import Path 
from tqdm import tqdm
import time 

1. Load the benign windows

Load each features.pt file from the dataset preparation, concatenate them (or keep them in a dataset that reads one file at a time). Each sample is one window of shape (32, 11).

In [2]:
BENIGN_WINDOWS = Path("..") / "Benign_windows"

train_files = sorted(BENIGN_WINDOWS.glob("*_train.pt"))
test_files = sorted(BENIGN_WINDOWS.glob("*_test.pt"))

train_windows = []
for pt_file in train_files:
    features = torch.load(pt_file, map_location="cpu", weights_only=False)["features"]
    train_windows.append(features)

test_windows = []
for pt_file in test_files:
    features = torch.load(pt_file, map_location="cpu", weights_only=False)["features"]
    test_windows.append(features)

print("Loaded train files:", len(train_files), "| test files:", len(test_files))

Loaded train files: 3 | test files: 3


In [3]:
class BenignWindowDataset(Dataset):
    def __init__(self, window_tensors):
        self.window_tensors = window_tensors
        self.window_counts = [tensor.shape[0] for tensor in window_tensors]
        self.total_windows = sum(self.window_counts)

    def __len__(self):
        return self.total_windows

    def __getitem__(self, index):
        start = 0
        for file_tensor, n_windows in zip(self.window_tensors, self.window_counts):
            if index < start + n_windows:
                local_index = index - start
                return file_tensor[local_index]
            start += n_windows

In [4]:
train_dataset = BenignWindowDataset(train_windows)
test_dataset = BenignWindowDataset(test_windows)

print("One train sample shape:", train_dataset[0].shape)
print("One test sample shape:", test_dataset[0].shape)

One train sample shape: torch.Size([32, 11])
One test sample shape: torch.Size([32, 11])


2. Build a DataLoader

A small dataset class that returns one window at a time, then a DataLoader with a batch size (64 or 128). Turning the saved tensors into training batches.

In [13]:
BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready | batch size:", BATCH_SIZE)
print("Train batches per epoch:", len(train_loader))

DataLoaders ready | batch size: 256
Train batches per epoch: 5216


3. Define the 30% Masking

For each batch:

- Sample a random mask over the time axis (length 32) with about 30% of positions masked.
- Keep a copy of the original window as the reconstruction target.
- - Replace masked time steps with a learned MASK embedding (a trainable vector of length 11), so masked positions are not confused with real zeros in the data.

Only the masked positions usually contribute to the loss (like BERT-style masked modeling).

In [14]:
MASK_PROPORTION = 0.30 

def mask_windows(batch, mask_embedding,mask_proportion=MASK_PROPORTION):
    target = batch.clone()
    time_mask = torch.rand(batch.shape[0], batch.shape[1], device=batch.device) < mask_proportion
    
    masked_batch = batch.clone()
    masked_batch[time_mask] = mask_embedding.to(batch.device)
    return masked_batch, target, time_mask

print("mask_windows ready | ratio:", MASK_PROPORTION)

mask_windows ready | ratio: 0.3


**mask_embedding** is a trainable vector of length 11 (one value per feature). At the start it is just random numbers, like the tensor printed above. During training, whenever a time step is masked, that whole step is replaced by this vector instead of zeros, so the model can tell “this position was hidden” from a real zero on the bus. The optimizer will update mask_embedding together with the network weights.

The function still returns:
- **masked_batch** — window with masked steps replaced by mask_embedding (model input)
- **target** — original window (reconstruction target)
- **time_mask** — **True** where a time step was hidden (used for the loss)

4. Define the 1D CNN Encoder + Reconstruction Head

Input: batch of windows (batch, 32, 11)  should be (batch, 11, 32) so that the Conv1D treats the features as channels and messages as the sequence length.

- Stack of 1D convolutions (and maybe pooling / residual blocks) that learn temporal patterns on the CAN bus.
- A small head that maps the encoder output back to the predicted features at each position (same shape as the input window).

The SSL objective is usually something like: minimize MSE (or similar) between predicted and true features only where the mask was applied.

In [15]:
class SSL_Encoder(nn.Module): 
    def __init__(self): 
        super().__init__()
        self.mask_embedding = nn.Parameter(torch.randn(11))
        self.conv1 = nn.Conv1d(in_channels=11, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.act = nn.ReLU()
        self.output = nn.Conv1d(in_channels=64, out_channels=11, kernel_size=3, padding=1)
        
    def forward(self, x): 
       # x starts as (batch, 32, 11)
        x = x.permute(0, 2, 1)   # become (batch, 11, 32) for Conv1d
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = self.output(x)
        x = x.permute(0, 2, 1)   # back to (batch, 32, 11)
        return x
    
model = SSL_Encoder()
print(model)
print("mask_embedding:", model.mask_embedding.data)

SSL_Encoder(
  (conv1): Conv1d(11, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (act): ReLU()
  (output): Conv1d(64, 11, kernel_size=(3,), stride=(1,), padding=(1,))
)
mask_embedding: tensor([-0.2721,  2.5466, -2.0894,  0.4817,  0.2605,  0.4700, -0.7133, -1.3215,
         0.5650,  0.0873, -1.7851])


nn.Conv1d expects:

(batch, channels, length)

- channels = different signal types measured at the same time (for you: the 11 features)
- length = the sequence the filter slides over (for you: the 32 messages)

The convolution slides a small filter along the last axis. So the last axis must be time.

- conv1: “What is happening in this small stretch of messages?”
- conv2: “How do those local patterns relate to the ones next to them?”

That gives the model a bit more capacity to learn normal CAN behaviour than a single layer, without making the network large.

5. Training Phase + Saving Model

For several epochs:

- Load batch → apply 30% mask → forward → loss on masked positions → backward → optimizer step.

Track the reconstruction loss. When it plateaus, save the encoder weights (not necessarily the reconstruction head), e.g. ssl_encoder.pt.

In [16]:
NUM_EPOCHS = 5
LEARNING_RATE = 0.001

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Device:", device)

Device: mps


In [17]:
def mask_loss(pred, target, time_mask): 
    pred_masked = pred[time_mask]
    true_masked = target[time_mask]
    return torch.mean((pred_masked - true_masked) ** 2)

print("mask_loss: MSE on masked time steps only (ignores visible positions)")

mask_loss: MSE on masked time steps only (ignores visible positions)


**time_mask** says which messages were hidden. We compute MSE, we only compare prediction vs target at the hidden steps. Getting the visible steps right does not count toward the loss.

In [18]:
model.train()
training_start = time.time()
print("Training started:", time.strftime("%Y-%m-%d %H:%M:%S"))

for epoch in range(NUM_EPOCHS):
    total_loss = 0
    n_batches = 0 
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        batch = batch.to(device)
        masked_batch, target, time_mask = mask_windows(batch, model.mask_embedding)
        pred = model(masked_batch)
        loss = mask_loss(pred, target, time_mask)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
        

    average_loss = total_loss / n_batches
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS} | average train loss: {average_loss:.6f}")
    
training_end = time.time()
print("Training completed:", time.strftime("%Y-%m-%d %H:%M:%S"))
print(f"Total training time: {(training_end - training_start) / 60:.2f} minutes")

Training started: 2026-07-19 02:10:07


Epoch 1/5: 100%|██████████| 5216/5216 [00:58<00:00, 88.45it/s] 


Epoch 1/5 | average train loss: 6931.709594


Epoch 2/5: 100%|██████████| 5216/5216 [00:38<00:00, 134.17it/s]


Epoch 2/5 | average train loss: 5721.648742


...

Epoch 3/5 | average train loss: 5525.072820


Epoch 4/5: 100%|██████████| 5216/5216 [00:40<00:00, 129.98it/s]


Epoch 4/5 | average train loss: 5408.731314


Epoch 5/5: 100%|██████████| 5216/5216 [00:37<00:00, 139.05it/s]

Epoch 5/5 | average train loss: 5334.475903
Training completed: 2026-07-19 02:13:41
Total training time: 3.56 minutes
